# Assignment 2: SVM using Machine Learning Library

Trong Assignment 2, mô hình SVM được huấn luyện bằng thư viện học máy thay vì tự cài đặt từ đầu. Bài này sử dụng `LinearSVC` từ scikit-learn để huấn luyện mô hình SVM tuyến tính trên bộ dữ liệu Chest X-Ray Images (Pneumonia).

Để so sánh công bằng với Assignment 1, dữ liệu được tiền xử lý giống nhau: ảnh được chuyển sang grayscale, resize về kích thước 128 × 128, chuẩn hóa pixel về khoảng [0, 1] và flatten thành vector 16.384 chiều.

Theo quy ước của Assignment 1, NORMAL được mã hóa là 1 và PNEUMONIA được mã hóa là -1. Khi tính Precision, Recall và F1-score, PNEUMONIA được xem là positive class.

In [6]:
import os
from data_utils import count_images, collect_data

BASE_DIR = r"C:\Users\Admin\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2\chest_xray\chest_xray"

print("BASE_DIR exists:", os.path.exists(BASE_DIR))
print("Train NORMAL exists:", os.path.exists(os.path.join(BASE_DIR, "train", "NORMAL")))
print("Train PNEUMONIA exists:", os.path.exists(os.path.join(BASE_DIR, "train", "PNEUMONIA")))

BASE_DIR exists: True
Train NORMAL exists: True
Train PNEUMONIA exists: True


In [7]:
train_counts = count_images(BASE_DIR, "train")
val_counts = count_images(BASE_DIR, "val")
test_counts = count_images(BASE_DIR, "test")

print("Train:", train_counts)
print("Validation:", val_counts)
print("Test:", test_counts)

total_images = sum(train_counts.values()) + sum(val_counts.values()) + sum(test_counts.values())
print("Total images:", total_images)

Train: {'NORMAL': 1341, 'PNEUMONIA': 3875}
Validation: {'NORMAL': 8, 'PNEUMONIA': 8}
Test: {'NORMAL': 234, 'PNEUMONIA': 390}
Total images: 5856


In [8]:
X_train, y_train = collect_data(BASE_DIR, "train")
X_val, y_val = collect_data(BASE_DIR, "val")
X_test, y_test = collect_data(BASE_DIR, "test")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

Loading train - NORMAL:   0%|          | 0/1342 [00:00<?, ?it/s]

Loading test - PNEUMONIA: 100%|██████████| 390/390 [00:02<00:00, 167.09it/s]

X_train shape: (5216, 16384)
y_train shape: (5216,)
X_val shape: (16, 16384)
y_val shape: (16,)
X_test shape: (624, 16384)
y_test shape: (624,)


Lưu kết quả Assignment 1 để so sánh:

In [9]:
implemented_svm_test_metrics = {
    "Accuracy": 0.7227564102448276,
    "Precision": 0.6954954954829641,
    "Recall": 0.9897435897182116,
    "F1": 0.8169312120663589
}

implemented_svm_confusion_matrix = [
    [65, 169],
    [4, 386]
]

Train SVM bằng thư viện:

In [ ]:
from sklearn.svm import LinearSVC

library_svm_model = LinearSVC(
    C=1.0,
    max_iter=10000,
    random_state=42
)

library_svm_model.fit(X_train, y_train)

Dự đoán

In [ ]:
y_val_pred_library = library_svm_model.predict(X_val)
y_test_pred_library = library_svm_model.predict(X_test)

Evaluate bằng sklearn:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_library_model(y_true, y_pred, pos_label=-1):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

Validation:

In [ ]:
library_svm_val_metrics = evaluate_library_model(
    y_val,
    y_val_pred_library,
    pos_label=-1
)

print("Validation metrics of Library SVM")

for metric_name, metric_value in library_svm_val_metrics.items():
    print(f"{metric_name}: {metric_value}")

Test:

In [ ]:
library_svm_test_metrics = evaluate_library_model(
    y_test,
    y_test_pred_library,
    pos_label=-1
)

print("Test metrics of Library SVM")

for metric_name, metric_value in library_svm_test_metrics.items():
    print(f"{metric_name}: {metric_value}")

Confusion matrix cho Assignment 2:

In [ ]:
import numpy as np

def confusion_matrix_binary(y_true, y_pred):
    normal_to_normal = np.sum((y_true == 1) & (y_pred == 1))
    normal_to_pneumonia = np.sum((y_true == 1) & (y_pred == -1))
    pneumonia_to_normal = np.sum((y_true == -1) & (y_pred == 1))
    pneumonia_to_pneumonia = np.sum((y_true == -1) & (y_pred == -1))

    return np.array([
        [normal_to_normal, normal_to_pneumonia],
        [pneumonia_to_normal, pneumonia_to_pneumonia]
    ])

library_svm_confusion_matrix = confusion_matrix_binary(
    y_test,
    y_test_pred_library
)

print(library_svm_confusion_matrix)

So sánh:

In [ ]:
import pandas as pd

comparison_data = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Implemented SVM": [
        implemented_svm_test_metrics["Accuracy"],
        implemented_svm_test_metrics["Precision"],
        implemented_svm_test_metrics["Recall"],
        implemented_svm_test_metrics["F1"]
    ],
    "Library SVM": [
        library_svm_test_metrics["Accuracy"],
        library_svm_test_metrics["Precision"],
        library_svm_test_metrics["Recall"],
        library_svm_test_metrics["F1"]
    ]
}

comparison_df = pd.DataFrame(comparison_data)

comparison_df

Vẽ biểu đồ:

In [ ]:
import matplotlib.pyplot as plt

comparison_df.plot(
    x="Metric",
    y=["Implemented SVM", "Library SVM"],
    kind="bar",
    figsize=(9, 5)
)

plt.ylabel("Score")
plt.title("Comparison between Implemented SVM and Library SVM")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

In hai confusion matrix cạnh nhau:

In [ ]:
print("Implemented SVM confusion matrix")
print(np.array(implemented_svm_confusion_matrix))

print("Library SVM confusion matrix")
print(library_svm_confusion_matrix)